# 실습 5-2 · Isolation Forest 학습 · 추론 · 평가 (비지도)

정답 없이 '다른 것과 다른 연결'을 찾습니다. 정답은 평가에만 씁니다.
1) 피처 구성 3가지 비교  2) 피처 묶음 하나씩 빼기  3) 경보 비율(contamination) 바꾸기  4) 이상 점수 상위 10건

지표 (발표자료 평가지표 장표와 같음): 정밀도 · 재현율 · F1 · F2 · AUC-ROC, 유형별 재현율(그 유형 공격 중 잡아낸 비율)

In [1]:
import os
import numpy as np
import pandas as pd

assert os.path.exists("../data/processed/train.csv"), "05a_data_preprocessing.ipynb를 먼저 실행하세요"
train = pd.read_csv("../data/processed/train.csv", dtype={"dsport": str})
test = pd.read_csv("../data/processed/test.csv", dtype={"dsport": str})
print(f"학습 {len(train)}건 (공격 {train['label'].sum()}) / 평가 {len(test)}건 (공격 {test['label'].sum()})")

학습 3520건 (공격 166) / 평가 1509건 (공격 71)


In [2]:
DESIGNED = ["log_ct_srv_dst", "log_ct_dst_ltm", "log_ct_src_dport_ltm",       # 연결 행위
            "log_sbytes_vs_svc", "log_dbytes_vs_svc", "log_rate_vs_svc",      # 규모 (서비스 평소 대비)
            "byte_ratio_vs_svc", "pkt_ratio_vs_svc"]                          # 방향 (서비스 평소 대비)
CATS = ["DoS", "Reconnaissance", "Exploits"]

In [6]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, fbeta_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

RAW = ["dur", "spkts", "dpkts", "sbytes", "dbytes", "rate", "ct_srv_dst", "ct_dst_ltm", "ct_src_dport_ltm"]
for d in (train, test):
    for c in RAW:
        d["log1p_" + c] = np.log1p(d[c])

def fit_predict(cols, contamination=0.05):
    """학습셋으로 표준화 · 모델 학습 → 평가셋 이상 점수와 경보(1)"""
    sc = StandardScaler().fit(train[cols])     #학습 데이터를 정규화(평균 0, 표준편차 1)
    model = IsolationForest(n_estimators=300, contamination=contamination, random_state=42)   #학습셋 이상 점수의 상위 5% 지점을 경보 기준선으로 정하는 값입니다. 
                                                                                              #실제 공격 비율(4.7%)과 비슷하게 맞춘 것이지, 정답을 본 것은 아닙니다.
    model.fit(sc.transform(train[cols]))       # 모델 학습 
    X = sc.transform(test[cols])               # 테스트 데이터셋 정규화 
    return -model.score_samples(X), (model.predict(X) == -1).astype(int)     # 점수(클수록 이상), 경보

def report(score, pred):
    y = test["label"]
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()     
    r = {"정밀도": precision_score(y, pred, zero_division=0), "재현율": recall_score(y, pred),
         "F1": fbeta_score(y, pred, beta=1, zero_division=0), "F2": fbeta_score(y, pred, beta=2, zero_division=0),
         "AUC-ROC": roc_auc_score(y, score), "경보": tp + fp, "오탐": fp, "미탐": fn}
    for c in CATS:                                   # 유형별 재현율
        r[c[:5] + " 재현율"] = pred[(test["attack_cat"] == c).values].mean()
    return r

## 1) 피처 구성 비교 (경보 비율 5% 고정)
A 원본 9개 그대로 / B 로그 변환 9개 / C 05a 설계 피처 8개

In [7]:
SETS = {"A 원본 9개": RAW, "B 로그 변환 9개": ["log1p_" + c for c in RAW], "C 설계 피처 8개": DESIGNED}
table = pd.DataFrame({name: report(*fit_predict(cols)) for name, cols in SETS.items()}).T
table.round(3)

,정밀도,재현율,F1,F2,AUC-ROC,경보,오탐,미탐,DoS 재현율,Recon 재현율,Explo 재현율
A 원본 9개,0.030,0.028,0.029,0.028,0.401,67.0,65.0,69.0,0.037,0.000,0.043
B 로그 변환 9개,0.051,0.056,0.053,0.055,0.827,79.0,75.0,67.0,0.111,0.000,0.043
C 설계 피처 8개,0.256,0.310,0.280,0.297,0.875,86.0,64.0,49.0,0.444,0.143,0.304


- 경보 수가 같을 때 정밀도 · 재현율이 높을수록 좋은 구성입니다.
- AUC-ROC는 경보 기준과 관계없이 '점수가 공격을 정상보다 위에 놓는 정도'입니다. 공격이 5%뿐이면 AUC-ROC가 0.87이어도 상위 5% 경보의 정밀도는 낮을 수 있습니다.

## 2) C에서 피처 묶음 하나씩 빼기: 재현율이 떨어지면 도움이 된 묶음, 오르면 이 데이터에서는 잡음

In [9]:
GROUPS = {"연결 행위": DESIGNED[:3], "규모": DESIGNED[3:6], "방향": DESIGNED[6:]}
rows = {"전체": report(*fit_predict(DESIGNED))}
for g, cols in GROUPS.items():
    rows[g + " 제외"] = report(*fit_predict([c for c in DESIGNED if c not in cols]))
pd.DataFrame(rows).T[["정밀도", "재현율", "AUC-ROC"]].round(3)

,정밀도,재현율,AUC-ROC
전체,0.256,0.310,0.875
연결 행위 제외,0.305,0.352,0.914
규모 제외,0.250,0.254,0.816
방향 제외,0.167,0.183,0.786


### 3) 이상 점수 상위 10건

In [11]:
score, pred = fit_predict(DESIGNED)    #score :얼마나 비정상적인지 나타내는 위험도 점수
test["score"] = score
test.sort_values("score", ascending=False).head(10)[
    ["id", "score", "service", "sbytes", "dbytes", "rate", "ct_dst_ltm", "attack_cat"]].round(3)

,id,score,service,sbytes,dbytes,rate,ct_dst_ltm,attack_cat
832,2553,0.706,dns,490,0,2000.000,1,Normal
467,2906,0.697,http,18578,1087890,953.975,9,Normal
815,3761,0.689,http,16914,1087890,491.669,1,Normal
1079,627,0.689,http,6910,268,92.393,1,Exploits
1067,1402,0.685,http,18578,1087890,513.708,7,Normal
338,19,0.684,http,2082,268,7.713,1,DoS
863,4614,0.684,http,18578,1087890,899.111,5,Normal
302,41,0.682,http,17850,1087890,55.585,14,Normal
1166,589,0.680,http,18890,1087890,950.986,3,Normal
776,2598,0.674,http,18682,1087890,231.167,1,Normal


### 토론
- A(원본 그대로)는 큰 정상 연결(대용량 웹 · 메일)을 이상으로 봐서 공격을 거의 못 찾습니다.
- 로그 변환(B)과 서비스 평소 대비 편차(C)를 거치면 같은 모델 · 같은 경보 수인데도 성능이 오릅니다.
- 그래도 C도 재현율이 낮습니다. 공격 상당수가 정상 웹 요청과 크기가 비슷합니다 → 비지도 하나로는 부족, 규칙 · 지도학습 · 사람 검토를 겹칩니다.